In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Agentic video understanding with Gemini

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/agentic-video/intro_agentic_video.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fgemini%2Fagentic-video%2Fintro_agentic_video.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/gemini/agentic-video/intro_agentic_video.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/agentic-video/intro_agentic_video.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/agentic-video/intro_agentic_video.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/agentic-video/intro_agentic_video.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/agentic-video/intro_agentic_video.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/agentic-video/intro_agentic_video.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/agentic-video/intro_agentic_video.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>

| Authors |
| --- |
| [Eric Dong](https://github.com/gericdong) |
| [Holt Skinner](https://github.com/holtskinner) |

## Overview

Traditional multimodal LLMs ingest video passively by extracting static frames at a fixed rate (1 FPS) alongside audio. For a 60-minute video, static ingestion consumes over 400,000 tokens—leading to high costs and multi-minute latency, even for simple verbal questions.

**Agentic Video** replaces passive frame extraction with an active **Think ➔ Act ➔ Observe** loop. Using a native timeline exploration tool, the model dynamically inspects only the required streams (captions/ASR, audio waveform, or visual frames), temporal windows, and sampling resolutions.

* **78%–96% Token Reduction**: Skips irrelevant visual frames (e.g., reducing a 1-hour video from ~407k to ~23k tokens).
* **Seconds Time-to-First-Byte (TTFB)**: Delivers rapid responses for transcript- and audio-answerable queries without decoding full video.
* **Sub-Second Precision (+1.1% to +4.6% QA, +17% Fast Action)**: Replaces uniform 1-FPS sampling with targeted high-FPS inspection (`fps=10+`).
* **Multi-Video Scalability**: Enables comparative analysis across multiple long-form videos within the 1M token context limit.

## Processing Modes & When to Use

| Mode | Description | Supported Models | Token Efficiency & Latency | When to Use |
| :--- | :--- | :--- | :--- | :--- |
| **Static**<br>*(default)* | Single-pass ingestion at fixed **1 FPS** (~300 tokens/sec) and **1 Kbps audio** with 1s timestamps. | All Gemini models | • **Baseline tokens:** ~300 tokens/sec<br>• **Latency:** Full ingestion required upfront | • Short clips (< 2 min)<br>• Requires simultaneous full audio & video<br>• Fixed-interval deterministic sampling<br>• Custom clipping (`start_offset`, `end_offset`) |
| **Agentic** | Iterative timeline navigation; dynamically loads frames and audio on-demand via reasoning budget. | • `gemini-3.7-flash`<br>• `gemini-3.6-flash`<br>• `gemini-3.5-flash-lite` | • **Tokens:** 70%–95% reduction on long content<br>• **Latency:** Fast TTFB (seconds) for transcript queries<br>• **Quality:** Higher overall answer accuracy | • Long videos > 2 min (lectures, meetings)<br>• Verbal / audio-first queries<br>• Localized visual search<br>• Fast-action clips needing high temporal precision<br>• Cross-video comparisons |

## Key Use Cases

| Use Case | Target Workloads | Why Static Fails | The Agentic Advantage |
| :--- | :--- | :--- | :--- |
| **Long-Form Q&A & Meetings** | • 30–90+ min meetings<br>• Lectures<br>• Earnings calls | • Decodes all 3,600+ frames (400k+ tokens) even for audio-only answers | • **78%–96% token reduction**<br>• **Seconds TTFB:** Uses transcript triage (`include_captions=True`) to bypass visual decoding |
| **Visual "Needle-in-a-Haystack"** | • Slide transitions<br>• Diagram edits<br>• Specific visual events | • Fixed 1 FPS dilutes attention<br>• Requires heavy external RAG pipelines | • **Coarse-to-fine zoom:** Probes at 0.5 FPS, then zooms into target 5s windows at 8–10 FPS<br>• **+3.5% to +4.6%** accuracy gain |
| **Fast-Action & Anomalies** | • Sports highlights<br>• Industrial QA<br>• UI glitches & telemetry | • 1 FPS misses sub-second actions (<1s)<br>• 10+ FPS full-video ingestion exceeds token limits | • **Adaptive High-FPS Replay:** Dynamically samples `fps=10+` only on critical 3–10s intervals<br>• **+17% accuracy** with 30% fewer tokens |
| **Cross-Video Synthesis** | • Multi-angle feeds<br>• Product reviews<br>• Lecture vs. lab footage | • Ingesting multiple 30+ min videos quickly exceeds the 1M token context limit | • **Multi-video feasibility:** Dynamically retrieves only relevant segments across files in a single prompt |

## Migration Guide: Upgrading to Agentic Video

1. **Remove Preprocessing Workarounds**: Eliminate custom ffmpeg downsampling (e.g., 0.1 FPS). Pass full video URIs directly with `media_processing="agentic"`.
2. **Configure `thinking_level`**:
   * `HIGH`: Dense visual QA, split-second action/sports analysis, or complex multi-step reasoning across 60+ minute videos.
   * `MEDIUM` (Default): General video Q&A, lecture summarization, and clip retrieval.
   * `LOW`: Fast transcript/caption searches and metadata extraction.

## 🚀 Getting Started & Setup

### Install Google Gen AI SDK for Python

Agentic video features require the Google Gen AI SDK for Python version `2.21.0` or later.

In [ ]:
%pip install --upgrade --quiet "google-genai[pyopenssl]>=2.21.0"

### Import Libraries

Import the required system and visual display components. We will also import the Pydantic library for validating structured outputs.

In [ ]:
import os
import sys

from IPython.display import Markdown, display
from google.genai import types

### Authenticate your Notebook Environment

If you are running this notebook in **Google Colab**, execute the cell below to authenticate.

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Set Google Cloud Project Information

To get started using Agent Platform, you must have an existing Google Cloud project and [enable the Agent Platform API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project](https://docs.cloud.google.com/resource-manager/docs/creating-managing-projects) and a [development environment](https://cloud.google.com/docs/authentication/set-up-adc-local-dev-environment).

In [ ]:
from google import genai

# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.getenv("GOOGLE_CLOUD_PROJECT"))

LOCATION = "global"

client = genai.Client(enterprise=True, project=PROJECT_ID, location=LOCATION)

### Set Model ID

Define `gemini-3.7-flash` as the target model for all tutorial examples.

Agentic video understanding is supported for Gemini 3.7 Flash, Gemini 3.6 Flash, and Gemini 3.5 Flash-Lite.

In [ ]:
# fmt: off
MODEL_ID = "gemini-3.7-flash"  # @param ["gemini-3.7-flash"] {type: "string"}
# fmt: on

## 🎥 Enable Agentic Video

When sending a video, set the `media_processing` field to `agentic`. Static processing is enabled by default.

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Part(
            file_data=types.FileData(
                file_uri="https://www.youtube.com/watch?v=LzExSq9DU9w",
                mime_type="video/mp4",
            ),
            media_processing="agentic",  # Options: "agentic", "static"
        ),
        "What were the key revenue figures mentioned by the presenter, and at what timestamp do they appear?",
    ],
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_level=types.ThinkingLevel.MEDIUM,  # LOW, MEDIUM, HIGH
        ),
    ),
)

display(Markdown(response.text.replace("$", "\\$")))
print(f"Total Token Count: {response.usage_metadata.total_token_count}")

For comparison, try the same video with static video processing. Then compare the total token count and processing time.

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Part(
            file_data=types.FileData(
                file_uri="https://www.youtube.com/watch?v=LzExSq9DU9w",
                mime_type="video/mp4",
            ),
            media_processing="static",  # Options: "agentic", "static"
        ),
        "What were the key revenue figures mentioned by the presenter, and at what timestamp do they appear?",
    ],
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_level=types.ThinkingLevel.MEDIUM,  # LOW, MEDIUM, HIGH
        ),
    ),
)

display(Markdown(response.text.replace("$", "\\$")))
print(f"Total Token Count: {response.usage_metadata.total_token_count}")

### Detailed timestamp-based descriptions

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Part(
            file_data=types.FileData(
                file_uri="https://storage.googleapis.com/generativeai-downloads/videos/Jukin_Trailcam_Videounderstanding.mp4",
                mime_type="video/mp4",
            ),
            media_processing="agentic",  # Options: "agentic", "static"
        ),
        "Describe what happens in this video in detail, with timestamps.",
    ],
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_level=types.ThinkingLevel.HIGH,  # LOW, MEDIUM, HIGH
        ),
    ),
)

display(Markdown(response.text.replace("$", "\\$")))
print(f"Total Token Count: {response.usage_metadata.total_token_count}")

### Youtube Shorts Video

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Part(
            file_data=types.FileData(
                file_uri="https://www.youtube.com/shorts/y-mrGw1wW8E",
                mime_type="video/mp4",
            ),
            media_processing="agentic",  # Options: "agentic", "static"
        ),
        "Explain this video",
    ],
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_level=types.ThinkingLevel.HIGH,  # LOW, MEDIUM, HIGH
        ),
    ),
)

display(Markdown(response.text.replace("$", "\\$")))
print(f"Total Token Count: {response.usage_metadata.total_token_count}")